# ARQ-FEC PoC Analysis

Analyses one case at a time: set `GEOMETRY` below and run the whole notebook.
Run it once per case to compare them side by side:

- **nofec**: control case, plain SCHC Ack-on-Error (RFC 8724, Compound ACK RFC 9441), no FEC
- **stream**: ARQ-FEC, Stream Encoding Geometry (XOR parity)
- **matrix**: ARQ-FEC, Matrix Encoding Geometry (Reed-Solomon)

**Three outcomes per iteration:**
- Delivered without retransmission (with FEC: FEC sufficed)
- ARQ rescued it (first pass not enough, retransmission fixed it)
- Failed (the device gave up: Sender-Abort, or the session timed out)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Load Data

Loads every CSV of the selected case. Use `FILES` to pick specific runs.

In [ ]:
GEOMETRY = "stream"   # "nofec" | "stream" | "matrix"
FILES = None          # or an explicit list, e.g. ["data/stats_2026-11-13_18-12-42.csv"]

PATTERN = f"data/stats-{GEOMETRY}_*.csv"

LABELS = {
    "nofec":  "No FEC (Ack-on-Error)",
    "stream": "ARQ-FEC Stream (XOR)",
    "matrix": "ARQ-FEC Matrix (RS)",
}
LABEL = LABELS[GEOMETRY]
FIRST_PASS = "No retx (no loss)" if GEOMETRY == "nofec" else "FEC sufficed"


def load_stats(files=None, pattern="data/stats_*.csv"):
    """Load one or more CSV files into a single DataFrame."""
    if files is None:
        files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError("No CSV files found.")

    dfs = []
    for f in files:
        d = pd.read_csv(f)
        d["source_file"] = f
        dfs.append(d)

    df = pd.concat(dfs, ignore_index=True)
    print(f"{LABEL}: loaded {len(df):,} iterations across {len(files)} file(s)")
    return df


df = load_stats(FILES, PATTERN)
df.head(3)

## 2. Derived Columns

Outcome, retransmission cost, and message counts per iteration.

- **Uplinks** (device → server): every Regular Fragment, All-1 and ACK-REQ sent
  (including the ones lost on the channel), plus the Sender-Abort.
- **Downlinks** (server → device): every SCHC ACK sent (C=0, C=1, and the matrix early ACK).

In [ ]:
# --- Outcome ---
# success_on_attempt: 0=failed, 1=no retransmission, 2=after retransmission(s)
cond = [
    (df["srv_rcs_ok"] == True) & (df["srv_retx_called"] == False),
    (df["srv_rcs_ok"] == True) & (df["srv_retx_called"] == True),
]
df["success_on_attempt"] = np.select(cond, [1, 2], default=0)

# --- Category ---
CATEGORIES = [FIRST_PASS, "ARQ rescued", "Failed"]
df["category"] = np.select(cond, CATEGORIES[:2], default="Failed")

# --- Retransmission cost ---
df["retx_frags"] = df["dev_regular_tx_attempts"] - df["dev_regular_count"]
df["actual_loss"] = df["dev_regular_lost"] / df["dev_regular_tx_attempts"]
df["all1_lost_scenario"] = df["dev_all1_lost"] > 0

# --- Messages ---
# dev_attempts = All-1 + ACK-REQ sent
df["uplinks"] = df["dev_regular_tx_attempts"] + df["dev_attempts"] + df["dev_aborted"].astype(int)
df["downlinks"] = df["srv_c0_sent"] + df["srv_c1_sent"] + df["srv_early_ack_sent"]

cols = ["iteration", "success_on_attempt", "category",
        "cfg_loss_probability", "cfg_allow_all1_loss", "cfg_max_attempts",
        "dev_regular_count", "dev_regular_tx_attempts", "retx_frags",
        "uplinks", "downlinks", "actual_loss", "all1_lost_scenario"]
df[cols].head(10)

## 3. Overall Summary

In [ ]:
total = len(df)
print(f"=== {LABEL} ===")
print(f"Total iterations        : {total}")
print(f"Config loss probability : {sorted(df['cfg_loss_probability'].unique())}")
print(f"Allow All-1 loss        : {df['cfg_allow_all1_loss'].unique()}")
print(f"MAX_ATTEMPTS            : {df['cfg_max_attempts'].unique()}")
print(f"SCHC packet / MTU       : {df['cfg_packet_bytes'].unique()} B / {df['cfg_mtu'].unique()} B")
print(f"Fragments per pass      : {df['dev_regular_count'].unique()}")
if df["cfg_loss_probability"].nunique() > 1:
    print()
    print("!! Several loss probabilities mixed: these totals average over them.")
    print("!! Use section 5 (per loss probability) for figures to present.")
print()
print(f"Success rate            : {df['success_on_attempt'].gt(0).mean():.1%}")
print(f"  └ No retransmission   : {(df['success_on_attempt']==1).mean():.1%}")
print(f"  └ After retx          : {(df['success_on_attempt']==2).mean():.1%}")
print(f"Failure rate            : {(df['success_on_attempt']==0).mean():.1%}")
print()
print(f"Avg uplinks per run     : {df['uplinks'].mean():.2f}")
print(f"Avg downlinks per run   : {df['downlinks'].mean():.2f}")
print()
print("--- Category breakdown ---")
print(df["category"].value_counts().reindex(CATEGORIES, fill_value=0).to_string())

## 4. FEC Effectiveness

**FEC sufficed** (no FEC: *No retx (no loss)*) = the packet was delivered from the
first pass alone, without any retransmission. With FEC, the parity recovered the
lost symbols; without FEC, this only happens when no fragment was lost.

**ARQ rescued** = the first pass was not enough; after a C=0, the retransmitted
tiles completed the packet.

**Failed** = the packet was not delivered within `MAX_ATTEMPTS`.

In [ ]:
cats = df["category"].value_counts().reindex(CATEGORIES, fill_value=0)
for label, count in cats.items():
    print(f"  {label:18s}  {count:4d}  ({count/total:6.1%})")

fig, ax = plt.subplots()
colors = {FIRST_PASS: "#2ecc71", "ARQ rescued": "#3498db", "Failed": "#e74c3c"}
bars = ax.bar(cats.index, cats.values, color=[colors[c] for c in cats.index],
              edgecolor="white", width=0.5)
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_ylabel("Iterations")
ax.set_title(f"{LABEL}: outcome (loss = {sorted(df['cfg_loss_probability'].unique())})")
sns.despine()
plt.tight_layout()
plt.show()

## 5. Breakdown by Loss Probability

In [ ]:
grouped = df.groupby("cfg_loss_probability").agg(
    total=("success_on_attempt", "count"),
    success_rate=("success_on_attempt", lambda x: x.gt(0).mean()),
    no_retx=("category", lambda x: (x == FIRST_PASS).mean()),
    arq_rescue=("category", lambda x: (x == "ARQ rescued").mean()),
    failed=("category", lambda x: (x == "Failed").mean()),
    avg_uplinks=("uplinks", "mean"),
    avg_downlinks=("downlinks", "mean"),
    avg_retx_frags=("retx_frags", "mean"),
    avg_actual_loss=("actual_loss", "mean"),
    all1_lost_rate=("all1_lost_scenario", "mean"),
).round(4)
print(LABEL)
grouped

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(LABEL)

# Success rate
axes[0].plot(grouped.index, grouped["success_rate"], "o-", color="#2ecc71")
axes[0].set_xlabel("Loss probability")
axes[0].set_ylabel("Success rate")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Success Rate")

# Stacked bar: no retx vs ARQ vs Failed
grouped[["no_retx", "arq_rescue", "failed"]].plot.bar(
    ax=axes[1], stacked=True,
    color=["#2ecc71", "#3498db", "#e74c3c"],
    edgecolor="white", legend=False
)
axes[1].legend([FIRST_PASS, "ARQ rescued", "Failed"], fontsize=8, loc="lower left")
axes[1].set_xlabel("Loss probability")
axes[1].set_ylabel("Ratio")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Outcome breakdown")

# Actual vs configured loss
axes[2].plot(grouped.index, grouped["avg_actual_loss"], "o-", color="#e67e22")
axes[2].plot(grouped.index, grouped.index, "--", color="gray", alpha=0.5, label="ideal")
axes[2].set_xlabel("Configured loss")
axes[2].set_ylabel("Actual loss")
axes[2].set_title("Loss simulation fidelity")
axes[2].legend()

sns.despine()
plt.tight_layout()
plt.show()

## 6. Downlinks (server → device)

Downlinks are scarce in LPWANs (duty cycle, receive windows). Every SCHC ACK
counts: C=0 (retransmission request), C=1 (session end), and, in the matrix
geometry only, the early "enough symbols" ACK. Without loss the minimum is one
downlink (C=1); FEC aims to keep it there as loss grows.

In [ ]:
downlinks = df.groupby("cfg_loss_probability").agg(
    avg_downlinks=("downlinks", "mean"),
    avg_c0=("srv_c0_sent", "mean"),
    avg_c1=("srv_c1_sent", "mean"),
    avg_early_ack=("srv_early_ack_sent", "mean"),
    max_downlinks=("downlinks", "max"),
    runs_with_c0=("srv_c0_sent", lambda x: x.gt(0).mean()),
).round(3)
print(LABEL)
display(downlinks)

fig, ax = plt.subplots()
downlinks[["avg_c1", "avg_early_ack", "avg_c0"]].plot.bar(
    ax=ax, stacked=True, color=["#2ecc71", "#95a5a6", "#e74c3c"], edgecolor="white"
)
ax.legend(["C=1 (end)", "Early ACK (matrix)", "C=0 (retx request)"], fontsize=8)
ax.set_xlabel("Loss probability")
ax.set_ylabel("Avg downlinks per run")
ax.set_title(f"{LABEL}: downlinks per run")
sns.despine()
plt.tight_layout()
plt.show()

## 7. All-1 Loss Scenarios

When the All-1 is lost, the server never gets the RCS (nor the residual bits).
The device times out and resends the All-1 (each All-1 / ACK-REQ sent
counts towards `MAX_ATTEMPTS`).

In [ ]:
all1_lost = df[df["all1_lost_scenario"]]
all1_ok = df[~df["all1_lost_scenario"]]

if len(all1_lost) > 0:
    print(f"{LABEL} - All-1 lost: {len(all1_lost)} / {len(df)} iterations")
    print()
    print("  All-1 never lost:")
    print(f"    Success: {all1_ok['success_on_attempt'].gt(0).mean():.1%}")
    print(f"    Retx needed: {all1_ok['srv_retx_called'].mean():.1%}")
    print()
    print("  All-1 lost at least once:")
    print(f"    Success: {all1_lost['success_on_attempt'].gt(0).mean():.1%}")
    print(f"    Retx needed: {all1_lost['srv_retx_called'].mean():.1%}")
    print()
    print("  All-1 loss scenario rows:")
    display(all1_lost[["iteration", "cfg_loss_probability",
                       "success_on_attempt", "category",
                       "srv_retx_called", "dev_attempts", "source_file"]])
else:
    print("No All-1 loss scenarios in this dataset.")
    print("(Set ALLOW_ALL1_LOSS = True with sufficient iterations to see them.)")

## 8. Uplink / Retransmission Overhead

Fragments per pass differ between cases (FEC adds parity; the packet and MTU may
differ too), so compare total uplinks, not only retransmissions.

In [ ]:
print(f"{LABEL} - average per iteration:")
print(f"  Fragments per pass     : {df['dev_regular_count'].mean():.1f}")
print(f"  Regular frags sent     : {df['dev_regular_tx_attempts'].mean():.1f}")
print(f"  Retransmitted frags    : {df['retx_frags'].mean():.1f}")
print(f"  Lost regular frags     : {df['dev_regular_lost'].mean():.1f}")
print(f"  All uplinks            : {df['uplinks'].mean():.1f}")
print()
print("By loss probability:")
overhead = df.groupby("cfg_loss_probability").agg({
    "dev_regular_count": "mean",
    "dev_regular_tx_attempts": "mean",
    "retx_frags": "mean",
    "dev_regular_lost": "mean",
    "uplinks": "mean",
}).round(2)
overhead

## 9. Failure Drill-Down

Rows where success_on_attempt == 0.

In [ ]:
failures = df[df["success_on_attempt"] == 0]
if len(failures) > 0:
    print(f"{LABEL} - failure count: {len(failures)}")
    display(failures[["iteration", "cfg_loss_probability",
                      "dev_aborted", "srv_timed_out", "dev_attempts",
                      "srv_all1_recvd", "srv_c0_sent", "all1_lost_scenario",
                      "dev_regular_lost", "dev_regular_tx_attempts",
                      "source_file"]])
else:
    print("No failures.")

## 10. Config Distribution

Quick check: what configurations are present in the loaded dataset?

In [ ]:
print(f"{LABEL} - configurations present:")
group_cols = ["cfg_loss_probability", "cfg_allow_all1_loss", "cfg_max_attempts",
              "cfg_packet_bytes", "cfg_mtu"]
configs = df.groupby(group_cols).size().reset_index(name="iterations")
configs

## 11. Source Files

Confirm which files contributed data.

In [ ]:
print(df["source_file"].value_counts().to_string())